In [18]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [19]:
words = open('names.txt','r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [20]:
len(words)

32033

In [21]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}


In [79]:
block_size = 3
X, Y = [], []
for w in words:
    context = [0]*block_size
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context),'---->', itos[ix])
        context = context[1:]+[ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

In [249]:
Y

tensor([ 5, 13, 13,  ..., 26, 24,  0])

In [251]:
#build the dataset
def build_dataset(words):
    block_size = 3
    X, Y = [], []
    for w in words:
        context = [0]*block_size
        for ch in w+'.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            #print(''.join(itos[i] for i in context),'---->', itos[ix])
            context = context[1:]+[ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape,Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [24]:
C = torch.randn((27,2))

In [25]:
emb = C[X]

In [26]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [27]:
h = torch.tanh(emb.view(-1,6) @ W1 +b1)
h


tensor([[ 0.9627, -0.9934,  0.9998,  ..., -0.9585,  0.9997, -0.3746],
        [ 0.9999, -0.9938,  0.9819,  ..., -0.9977,  0.9985, -0.3263],
        [ 0.9164, -0.6967,  0.1788,  ..., -0.9674,  0.8426, -0.4725],
        ...,
        [-0.3944,  0.0406,  0.9971,  ..., -0.5570,  0.9293, -0.9510],
        [-0.0832, -0.9701,  0.9623,  ..., -0.9857,  0.8340,  0.1198],
        [ 0.9962, -0.8776,  0.1008,  ..., -0.9993,  0.9075, -0.6434]])

In [28]:
#torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]], 1)
#torch.cat(torch.unbind(emb, 1),1) == emb.view(32,6)

In [29]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [30]:
logits = h @ W2 + b2

In [31]:
counts = logits.exp()

In [32]:
probs = counts / counts.sum(1,keepdims=True)

In [39]:
loss=-probs[torch.arange(32),Y].log().mean() # Negative log likelihood loss
loss

tensor(18.3281)

In [40]:
#--------------Formatted------------------

In [252]:
Xtr.shape, Ytr.shape

(torch.Size([182625, 3]), torch.Size([182625]))

In [257]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2),generator=g)
W1 = torch.randn((6,100),generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27),generator=g)
b2 = torch.randn(27,generator=g)
parameters = [C,W1,b1,W2,b2]

In [258]:
sum(p.nelement() for p in parameters) # Number of parameters

3481

In [259]:
for p in parameters:
    p.requires_grad = True

In [260]:
lre = torch.linspace(-3,0,1000)
lrs = 10**lre

In [261]:
    #counts = logits.exp()
    #prob = counts/ counts.sum(1, keepdims=True)
    #loss = -prob[torch.arange(32), Y].log().mean()
    #loss = F.cross_entropy(logits, Y[ix]) # Optimized way to do the above three lines operation [Fused kernels] [Optimized for Forward/backward pass]

In [264]:
# lri = []
# lossi = []
for i in range(10000):
    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0],(32,))

    # Forward Pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix]) # exp,log,mean, -ve [Negative log likelihood loss]
    
    # Backward Pass
    for p in parameters:
        p.grad=None
    loss.backward()
    
    #update
    # lr = lrs[i]
    lr = 0.1
    for p in parameters:
        p.data += -lr*p.grad
    
    # Track Learning Rate
    # lri.append(lre[i])
    # lossi.append(loss.item())

print(loss.item())


2.3683104515075684


In [224]:
# plt.plot(lri, lossi)

In [265]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev) # exp,log,mean, -ve [Negative log likelihood loss]
loss    

tensor(2.4167, grad_fn=<NllLossBackward0>)

In [248]:
#dataset should be divided into three splits which are: Training split [Train Parameters], Dev/Validation split [Train Hyperparameters], Test split [Loss evaluation]
# 80%,10%,10%